# 04. Матрицы сходства и совместной встречаемости

Строим TF-IDF cosine similarity между топ-запросами, heatmap query×brand и сходство названий брендов.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_utils import (
    apply_plot_style, save_fig, ensure_dirs, load_query_clicks, load_sku_desc,
    text_len, parquet_num_rows, parquet_schema_names, dataset_overview_stats,
    save_stats, QUERY_CLICKS_PATH, SKU_DESC_PATH, SKUS_PKL_PATH,
    MVIDEO_RED, DARK_SLATE, MUTED,
)
ensure_dirs()
apply_plot_style()
pd.set_option("display.max_colwidth", 80)
print("ROOT:", ROOT)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
df = load_query_clicks(n=400_000, seed=42)
print(df.shape)


## Heatmap: запрос × бренд


In [ ]:
top_q = df["query_text"].value_counts().head(15).index
top_b = df["sku_brand_name"].replace("", np.nan).dropna().value_counts().head(15).index
sub = df[df["query_text"].isin(top_q) & df["sku_brand_name"].isin(top_b)]
mat = pd.crosstab(sub["query_text"], sub["sku_brand_name"]).reindex(index=top_q, columns=top_b, fill_value=0)
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(np.log1p(mat), cmap=sns.light_palette(MVIDEO_RED, as_cmap=True), ax=ax, linewidths=0.3, linecolor="white", cbar_kws={"label": "log(1 + частота)"})
ax.set_title("Совместная встречаемость: запрос × бренд")
ax.set_xlabel("Бренд"); ax.set_ylabel("Запрос")
plt.xticks(rotation=45, ha="right")
save_fig(fig, "06_query_brand_heatmap.png"); plt.show()


## TF-IDF сходство топ-запросов


In [ ]:
queries = df["query_text"].value_counts().head(25).index.tolist()
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
sim = cosine_similarity(vec.fit_transform(queries))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(sim, xticklabels=queries, yticklabels=queries, cmap=sns.light_palette(MVIDEO_RED, as_cmap=True), vmin=0, vmax=1, square=True, ax=ax, cbar_kws={"label": "Косинусное сходство (TF-IDF)"})
ax.set_title("Матрица сходства топ-запросов (TF-IDF char n-grams)")
plt.xticks(rotation=55, ha="right", fontsize=8); plt.yticks(fontsize=8)
save_fig(fig, "07_tfidf_similarity_heatmap.png"); plt.show()


## Сходство названий брендов


In [ ]:
brands = df["sku_brand_name"].replace("", np.nan).dropna().value_counts().head(20).index.tolist()
sim_b = cosine_similarity(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(brands))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_b, xticklabels=brands, yticklabels=brands, cmap=sns.light_palette(DARK_SLATE, as_cmap=True), vmin=0, vmax=1, square=True, ax=ax, cbar_kws={"label": "Сходство названий брендов"})
ax.set_title("TF-IDF сходство названий топ-брендов")
plt.xticks(rotation=45, ha="right", fontsize=8); plt.yticks(fontsize=8)
save_fig(fig, "14_brand_name_similarity.png"); plt.show()


## Co-occurrence: запрос × subject


In [ ]:
top_subj = df["sku_subject_id"].value_counts().head(12).index
sub2 = df[df["query_text"].isin(top_q) & df["sku_subject_id"].isin(top_subj)]
mat2 = pd.crosstab(sub2["query_text"], sub2["sku_subject_id"].astype(str))
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(np.log1p(mat2), cmap=sns.light_palette(DARK_SLATE, as_cmap=True), ax=ax, linewidths=0.3, linecolor="white", cbar_kws={"label": "log(1 + частота)"})
ax.set_title("Совместная встречаемость: запрос × subject_id")
save_fig(fig, "18_query_subject_heatmap.png"); plt.show()
